In [1]:
import pandas as pd
import yfinance as yf

# Step 1: Getting all required Data
- Download relevant Data of the top 50 performing stocks in the ASX 50 using Yahoo Finance
    - Get 5 Year data or whatever is the Maximum data we can get
    - Also wanted the Top 10 ETF in betashares, the top 10 was choosen by Claude, not using any maths or calculations

In [2]:
ticker_list = [
    "CBA.AX", "BHP.AX", "CSL.AX", "NAB.AX", "WBC.AX",
    "ANZ.AX", "WES.AX", "MQG.AX", "RIO.AX", "WOW.AX",
    "FMG.AX", "XRO.AX", "TLS.AX", "REA.AX", "STO.AX",
    "WDS.AX", "COL.AX", "RMD.AX", "ALL.AX", "GMG.AX",
    "TCL.AX", "NST.AX", "QBE.AX", "SHL.AX", "COH.AX",
    "IAG.AX", "MPL.AX", "ORG.AX", "AGL.AX", "APA.AX",
    "AMC.AX", "BXB.AX", "SEK.AX", "ORI.AX", "MIN.AX",
    "JHX.AX", "S32.AX", "ALD.AX", "CPU.AX", "ASX.AX",
    "SOL.AX", "WBC.AX", "PPT.AX", "AZJ.AX", "DXS.AX",
    "GPT.AX", "MGR.AX", "SGP.AX", "VCX.AX", "CHC.AX"
]

In [3]:
ticker_list[:5]  # Display the first 5 tickers to verify the format

['CBA.AX', 'BHP.AX', 'CSL.AX', 'NAB.AX', 'WBC.AX']

In [4]:
len(ticker_list)  # Check the total number of tickers

50

In [5]:
DOWNLOAD_DATA = False  # Set to False if we already downloaded

In [6]:
if DOWNLOAD_DATA:
    for ticker in ticker_list:
        try:
            print(f"Downloading data for {ticker}...")
            data = yf.download(ticker, period="5y", interval="1d")
            data = data[["Close", "Volume"]]
            data.to_csv(f"stock_data/{ticker}_data.csv")
            print(f"Data for {ticker} saved to stock_data/{ticker}_data.csv")
        except Exception as e:
            print(f"Error downloading data for {ticker}: {e}")

In [7]:
etf_list = [
    "A200.AX",   # 1. Australia 200 ETF              — AUM: $9.70B
    "NDQ.AX",    # 2. Nasdaq 100 ETF                 — AUM: $7.27B
    "AAA.AX",    # 3. Australian High Interest Cash   — AUM: $5.13B
    "BGBL.AX",   # 4. Global Shares ETF               — AUM: $3.73B
    "ETHI.AX",   # 5. Global Sustainability Leaders   — AUM: $3.48B
    "HBRD.AX",   # 6. Australian High Interest Bond   — AUM: $2.51B
    "HGBL.AX",   # 7. Global Shares Currency Hedged   — AUM: $2.30B
    "QPON.AX",   # 8. Bank Senior Floating Rate Bond  — AUM: $1.95B
    "CRED.AX",   # 9. Investment Grade Corporate Bond — AUM: $1.77B
    "QAU.AX",    # 10. Gold Bullion Currency Hedged   — AUM: $1.54B
]

if DOWNLOAD_DATA:
    for etf in etf_list:
        try:
            print(f"Downloading data for {etf}...")
            data = yf.download(etf, period="5y", interval="1d")
            data = data[["Close", "Volume"]]
            data.to_csv(f"stock_data/{etf}_data.csv")
            print(f"Data for {etf} saved to stock_data/{etf}_data.csv")
        except Exception as e:
            print(f"Error downloading data for {etf}: {e}")

# Step 2: Get Log Returns of each tickers
- Make a returns matrix of each tickers, for every trading day
- Preprocess the data to handle missing values and etc.

In [8]:
import numpy as np

In [9]:
# Preprocess df because its weirdly formatted
def preprocess_data(file_path):
    # Skip first 2 rows
    df = pd.read_csv(file_path, skiprows=2)
    
    # Rename columns
    df.columns = ["Date", "Close", "Volume"]

    #Convert Date to datetime
    df["Date"] = pd.to_datetime(df["Date"])

    # Turn Close and Volumne into numeric
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce") 
    return df

PREPROCESS = False  # Set to False if we already preprocessed
if PREPROCESS:
    for ticker in ticker_list+etf_list:
        file_path = f"stock_data/{ticker}_data.csv"
        try:
            df = preprocess_data(file_path)
            df.to_csv(f"stock_data/{ticker}_data.csv", index=False)
        except Exception as e:
            print(f"Error preprocessing data for {ticker}: {e}")


In [10]:
# Makes a Log return Data Frame
return_df = pd.DataFrame()

for ticker in [ticker_list, etf_list]:
    for symbol in ticker:
        try:
            data = pd.read_csv(f"stock_data/{symbol}_data.csv", index_col="Date", parse_dates=True)

            data[f"{symbol}_Return"] = np.log(data["Close"]).diff() # Diff to get daily returns
            data = data.iloc[1:]  # Drop the first row which will be NaN after diff

            # Add the return column to the return_df
            return_df = pd.concat([return_df, data[f"{symbol}_Return"]], axis=1)
        except Exception as e:
            print(f"Error processing data for {symbol}: {e}")

In [11]:
return_df.head()

,CBA.AX_Return,BHP.AX_Return,CSL.AX_Return,NAB.AX_Return,WBC.AX_Return,ANZ.AX_Return,WES.AX_Return,MQG.AX_Return,RIO.AX_Return,WOW.AX_Return,...,A200.AX_Return,NDQ.AX_Return,AAA.AX_Return,BGBL.AX_Return,ETHI.AX_Return,HBRD.AX_Return,HGBL.AX_Return,QPON.AX_Return,CRED.AX_Return,QAU.AX_Return
2021-05-04,0.008327,0.025195,0.000959,0.000734,NaN,-0.009322,0.007226,0.000561,0.024999,0.006135,...,0.005050,-0.010807,-0.0001,NaN,-0.005106,0.000000,NaN,-0.001151,0.000371,0.006526
2021-05-05,0.024787,0.010929,0.023504,0.004394,NaN,-0.032790,0.000554,0.005165,0.011386,0.003053,...,0.004356,-0.011956,0.0000,NaN,-0.005132,0.000000,NaN,-0.000384,0.000372,-0.004742
2021-05-06,0.002370,0.021104,-0.002777,-0.030041,0.002308,-0.009001,-0.006107,-0.013183,0.009748,-0.002544,...,-0.004188,-0.006551,0.0000,NaN,-0.001717,-0.001945,NaN,0.000384,0.001112,0.007105
2021-05-07,0.010489,0.005807,-0.008633,0.008249,0.002687,0.003610,0.007213,-0.003465,0.010678,0.004067,...,0.002264,0.004142,0.0002,NaN,0.000859,0.001945,NaN,0.000000,0.001852,0.016384
2021-05-10,0.013011,0.030669,0.003092,0.011509,0.012569,0.012126,0.010815,-0.001516,0.044845,0.027029,...,0.012814,-0.003796,-0.0002,NaN,0.000000,-0.003895,NaN,-0.000384,-0.001852,0.009243


*Note*:\
Here we notice that are some incomplete Data, I understand this when downloading. However, now I notice that it would be difficult to fix, and filling in the missing data would cause errors and bias for high numbers of incompletion. 

So, I will be taking only data where there are $\le 5$% missing data, and from the remaining columns fill missing values based on mean returns

In [ ]:
# Delete columns with more than 5% missing values
threshold = 0.05
col = return_df.isnull().sum()/len(return_df)<=threshold
return_df = return_df.loc[:, col]

In [25]:
for ticker, i in return_df.isnull().sum().items():
    if i > 0:
        print(ticker, i)

WBC.AX_Return 2
WBC.AX_Return 2
A200.AX_Return 1
AAA.AX_Return 1
HBRD.AX_Return 1
QPON.AX_Return 1
CRED.AX_Return 1
QAU.AX_Return 1


*Note*:\
Here we can conclude that there may just be errors in the data collection.

Therefore We will simply fill the missing values with 0, this won't affect the data too much considering only 1,2 data are missing in more than 1,000 data.

Another choice: Use ffill on original data, this would allow us to capture if they are too many consecutive missing values, however, we know that there are barely any missing values, and manually changing the original data takes too long.

In [26]:
return_df.fillna(0, inplace=True)

,CBA.AX_Return,BHP.AX_Return,CSL.AX_Return,NAB.AX_Return,WBC.AX_Return,ANZ.AX_Return,WES.AX_Return,MQG.AX_Return,RIO.AX_Return,WOW.AX_Return,...,VCX.AX_Return,CHC.AX_Return,A200.AX_Return,NDQ.AX_Return,AAA.AX_Return,ETHI.AX_Return,HBRD.AX_Return,QPON.AX_Return,CRED.AX_Return,QAU.AX_Return
2021-05-04,0.008327,0.025195,0.000959,0.000734,0.000000,-0.009322,0.007226,0.000561,0.024999,0.006135,...,-0.009449,-0.002817,0.005050,-0.010807,-0.000100,-0.005106,0.000000,-0.001151,0.000371,0.006526
2021-05-05,0.024787,0.010929,0.023504,0.004394,0.000000,-0.032790,0.000554,0.005165,0.011386,0.003053,...,0.000000,0.015396,0.004356,-0.011956,0.000000,-0.005132,0.000000,-0.000384,0.000372,-0.004742
2021-05-06,0.002370,0.021104,-0.002777,-0.030041,0.002308,-0.009001,-0.006107,-0.013183,0.009748,-0.002544,...,-0.025642,-0.022473,-0.004188,-0.006551,0.000000,-0.001717,-0.001945,0.000384,0.001112,0.007105
2021-05-07,0.010489,0.005807,-0.008633,0.008249,0.002687,0.003610,0.007213,-0.003465,0.010678,0.004067,...,0.000000,0.007782,0.002264,0.004142,0.000200,0.000859,0.001945,0.000000,0.001852,0.016384
2021-05-10,0.013011,0.030669,0.003092,0.011509,0.012569,0.012126,0.010815,-0.001516,0.044845,0.027029,...,0.003241,0.018157,0.012814,-0.003796,-0.000200,0.000000,-0.003895,-0.000384,-0.001852,0.009243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-27,-0.008114,0.001069,0.013903,0.002990,-0.000769,-0.008037,-0.004593,-0.001120,0.007779,-0.005292,...,-0.015748,-0.005013,-0.002661,0.009115,0.000199,-0.003244,0.001982,0.000000,-0.001326,0.006165
2026-04-28,0.008801,-0.013084,-0.022401,-0.000747,-0.009278,0.002501,-0.021209,-0.000690,-0.004695,-0.005054,...,-0.011976,-0.004533,-0.005276,-0.004636,0.000000,-0.001951,-0.000991,0.000381,-0.001328,-0.015892
2026-04-29,-0.014015,-0.008697,-0.024503,-0.012525,-0.010411,0.004155,0.000415,0.005463,-0.007699,-0.005616,...,0.000000,0.004030,-0.003717,-0.000179,0.000000,-0.004568,0.000991,-0.000761,0.002212,-0.009823
2026-04-30,0.008559,-0.022638,-0.011273,0.005280,0.007299,0.012907,0.007986,0.008330,-0.020107,-0.080959,...,0.008000,0.012987,-0.002002,0.003034,0.000398,-0.001309,-0.000991,0.000000,-0.004429,-0.006051


# Steo 3: Get mean, cov, plot correlation (EDA)